**Navigation** : [carnet source Python](Jauge-02-Invariance-Jauge-U1-SU2-Python.ipynb) | [Index de la série](README.md)

# De la symétrie globale à la jauge locale : U(1) puis SU(2) (C#)

**Série** : Physique / Jauge · **Noyau** : .NET (C#)

Jumeau C# de [Jauge-02 — De la symétrie globale à la jauge locale : U(1) puis SU(2) (Python)](Jauge-02-Invariance-Jauge-U1-SU2-Python.ipynb) : même fil pédagogique, mêmes verdicts — rejoués ici en **.NET natif**, BCL seule, aucun paquet externe ni NuGet. Pour que tout reste rationnel, les champs y sont des polynômes de $(t, x)$.

[Jauge-01](Jauge-01-Theoreme-Noether-Python.ipynb) a établi le pont symétrie globale → quantité conservée (théorème de Noether). Ce carnet franchit l'étape suivante de l'arc : que se passe-t-il quand on exige la symétrie **en chaque point de l'espace-temps séparément** ?

C'est l'idée de **jauge locale**, introduite pour le spin isotopique par Yang et Mills (1954) par analogie explicite avec la phase du champ chargé (copie archivée au gisement, cf. Références) :

1. **U(1)** — un champ scalaire complexe est invariant sous une phase *constante* ; rendre la phase *locale* $\alpha(x,t)$ casse l'invariance, et la restaurer force l'introduction d'un champ de jauge $A_\mu$ via la dérivée covariante $D_\mu$ ;
2. **SU(2)** — la version non abélienne de Yang–Mills : les champs de jauge eux-mêmes portent la charge, le tenseur de courbure gagne un terme d'**auto-interaction** $\varepsilon^{abc} W^b W^c$, vérifié ici par le calcul exact du commutateur $[D_\mu, D_\nu]$ ;
3. **Contrôles** — invariance **exacte** de $\mathrm{Tr}(F_{\mu\nu} F_{\mu\nu}^\dagger)$ sous une transformation de jauge SU(2) explicite, et contre-témoins qui montrent que le calcul détecte une transformation fausse.

**L'outil.** Ni flottant, ni bibliothèque de calcul formel : chaque quantité est un polynôme en $(t, x)$ à coefficients rationnels exacts (entiers de taille arbitraire, fournis par `System.Numerics`), ou un quotient de deux tels polynômes. Une identité est établie quand le reste calculé est le polynôme **nul**, et l'assertion `Exige` arrête le carnet sinon.

**Filière** : SymmetryGroups/Jauge — prolonge [Jauge-01](Jauge-01-Theoreme-Noether-Python.ipynb).

## 1. Symétrie U(1) globale d'un champ scalaire complexe

Champ complexe $\varphi = u + iv$ en dimension $1+1$ ($t, x$), Lagrangien du champ libre :

$$\mathcal{L} = \partial_\mu \varphi^* \, \partial^\mu \varphi - m^2 \varphi^* \varphi .$$

La transformation $\varphi \to e^{i\alpha}\varphi$ avec $\alpha$ **constant** est la symétrie de jauge globale. On la vérifie exactement en substituant $u \to u\cos\alpha - v\sin\alpha$, $v \to u\sin\alpha + v\cos\alpha$.

En C#, le calcul tourne sur une arithmétique écrite pour ce carnet, sans paquet. La cellule de code qui suit la définit en entier :

- `Rat` : un rationnel réduit (deux `BigInteger`) ; `Cpx` : un nombre complexe (deux rationnels) ;
- `Poly` : un polynôme en $(t, x)$ à coefficients complexes, pouvant aussi contenir des paramètres constants $m$, $q$, $g$ ; `Frac` : le quotient de deux polynômes ;
- `M2` : les matrices $2\times 2$, comme tableaux imbriqués de `Frac` ; `Pt` : les mêmes, évaluées en un point.

Pour rester dans les quotients, la phase est décrite par sa demi-tangente $s = \tan(\alpha/2)$ :

$$e^{i\alpha} = \frac{1+is}{1-is}.$$

Les champs $u$ et $v$ sont des polynômes explicites de $(t, x)$ ; la masse $m$ et la demi-tangente $s$ restent des paramètres libres. L'identité vaut donc pour toute masse et toute phase constante.

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;
using System.Numerics;

// ==========================================================================
// Arithmetique exacte, BCL seule (System.Numerics.BigInteger).
//   Rat   rationnel reduit (denominateur > 0, pgcd 1)
//   Cpx   complexe = paire de rationnels
//   Poly  polynome a coefficients Cpx en (t, x) et en parametres constants
//   Frac  fraction rationnelle de polynomes
//   M2    matrices 2x2 : tableaux imbriques de Frac
//   Pt    matrices evaluees en un point : tableaux imbriques de Cpx
// Aucun flottant : une identite est soit etablie (reste nul), soit pas.
// ==========================================================================

// ---------- Rationnel exact ----------
readonly struct Rat : IEquatable<Rat>
{
    public BigInteger Num { get; }
    public BigInteger Den { get; }

    public Rat(long num, long den) : this((BigInteger)num, (BigInteger)den) { }

    public Rat(BigInteger num, BigInteger den)
    {
        if (den.IsZero) throw new DivideByZeroException("rationnel a denominateur nul");
        if (den.Sign < 0) { num = -num; den = -den; }
        var pgcd = BigInteger.GreatestCommonDivisor(num, den);
        if (pgcd > 1) { num /= pgcd; den /= pgcd; }
        Num = num; Den = den;
    }

    public static readonly Rat Zero = new Rat(0, 1);
    public static readonly Rat One = new Rat(1, 1);
    public static implicit operator Rat(long v) => new Rat(v, 1);

    public static Rat operator +(Rat a, Rat b) => new Rat(a.Num * b.Den + b.Num * a.Den, a.Den * b.Den);
    public static Rat operator -(Rat a, Rat b) => new Rat(a.Num * b.Den - b.Num * a.Den, a.Den * b.Den);
    public static Rat operator -(Rat a) => new Rat(-a.Num, a.Den);
    public static Rat operator *(Rat a, Rat b) => new Rat(a.Num * b.Num, a.Den * b.Den);
    public static Rat operator /(Rat a, Rat b) => new Rat(a.Num * b.Den, a.Den * b.Num);
    public static Rat operator /(Rat a, long n) => new Rat(a.Num, a.Den * n);

    public bool Equals(Rat autre) => Num == autre.Num && Den == autre.Den;
    public override bool Equals(object objet) => objet is Rat autre && Equals(autre);
    public override int GetHashCode() => Num.GetHashCode() * 31 + Den.GetHashCode();
    public static bool operator ==(Rat a, Rat b) => a.Equals(b);
    public static bool operator !=(Rat a, Rat b) => !a.Equals(b);
    public static bool operator <(Rat a, Rat b) => a.Num * b.Den < b.Num * a.Den;
    public static bool operator >(Rat a, Rat b) => a.Num * b.Den > b.Num * a.Den;
    public static bool operator <=(Rat a, Rat b) => !(a > b);
    public static bool operator >=(Rat a, Rat b) => !(a < b);

    public bool EstNul => Num.IsZero;
    public Rat Abs => Num.Sign < 0 ? -this : this;
    public override string ToString() => Den.IsOne ? Num.ToString() : Num + "/" + Den;

    public static string Ecrit(Rat r) => r.Den.IsOne ? r.ToString() : "(" + r + ")";

    // Racine carree approchee (huit decimales), pour afficher un ecart compare
    // a un seuil : (isqrt(n d 10^16) / (d 10^8))^2 = n/d a l'arrondi pres.
    public Rat Racine()
        => new Rat(BigIntegerSqrt(Num * Den * BigInteger.Pow(10, 16)), Den * BigInteger.Pow(10, 8));

    // Racine carree entiere (Newton, depart par puissance de deux).
    static BigInteger BigIntegerSqrt(BigInteger n)
    {
        if (n.Sign <= 0) return BigInteger.Zero;
        var x = BigInteger.One << (int)((n.GetBitLength() + 1) / 2);
        while (true)
        {
            var y = (x + n / x) >> 1;
            if (y >= x) return x;
            x = y;
        }
    }

    // Ecriture scientifique a `chiffres` chiffres significatifs, arrondis, pour
    // se lire comme la source Python (3.92e-15). La valeur reste exacte :
    // l'arrondi n'affecte que l'affichage.
    public string Sci(int chiffres = 3)
    {
        if (EstNul) return "0." + new string('0', chiffres - 1) + "e+00";
        var n = BigInteger.Abs(Num);
        var d = Den;
        int e = n.ToString().Length - d.ToString().Length;
        if (e >= 0 ? n < d * BigInteger.Pow(10, e) : n * BigInteger.Pow(10, -e) < d) e--;
        int k = chiffres - 1 - e;
        var numerateur = k >= 0 ? n * BigInteger.Pow(10, k) : n;
        var denominateur = k >= 0 ? d : d * BigInteger.Pow(10, -k);
        var m = numerateur / denominateur;
        if (2 * (numerateur % denominateur) >= denominateur) m++;
        if (m == BigInteger.Pow(10, chiffres)) { m /= 10; e++; }
        var texte = m.ToString();
        return texte[..1] + "." + texte[1..] + "e" + (e < 0 ? "-" : "+") + Math.Abs(e).ToString("00");
    }
}

// ---------- Complexe exact : paire de rationnels ----------
readonly struct Cpx : IEquatable<Cpx>
{
    public Rat Re { get; }
    public Rat Im { get; }
    public Cpx(Rat re, Rat im) { Re = re; Im = im; }

    public static readonly Cpx Zero = new Cpx(Rat.Zero, Rat.Zero);
    public static readonly Cpx Un = new Cpx(Rat.One, Rat.Zero);
    public static readonly Cpx I = new Cpx(Rat.Zero, Rat.One);
    public static implicit operator Cpx(Rat r) => new Cpx(r, Rat.Zero);
    public static implicit operator Cpx(long v) => new Cpx(v, Rat.Zero);

    public static Cpx operator +(Cpx a, Cpx b) => new Cpx(a.Re + b.Re, a.Im + b.Im);
    public static Cpx operator -(Cpx a, Cpx b) => new Cpx(a.Re - b.Re, a.Im - b.Im);
    public static Cpx operator -(Cpx a) => new Cpx(-a.Re, -a.Im);
    public static Cpx operator *(Cpx a, Cpx b)
        => new Cpx(a.Re * b.Re - a.Im * b.Im, a.Re * b.Im + a.Im * b.Re);
    public static Cpx operator /(Cpx a, Cpx b)
    {
        var n = b.Norme2;
        return new Cpx((a.Re * b.Re + a.Im * b.Im) / n, (a.Im * b.Re - a.Re * b.Im) / n);
    }

    public bool Equals(Cpx autre) => Re == autre.Re && Im == autre.Im;
    public override bool Equals(object objet) => objet is Cpx autre && Equals(autre);
    public override int GetHashCode() => Re.GetHashCode() * 31 + Im.GetHashCode();
    public static bool operator ==(Cpx a, Cpx b) => a.Equals(b);
    public static bool operator !=(Cpx a, Cpx b) => !a.Equals(b);

    public Cpx Conj => new Cpx(Re, -Im);
    public Rat Norme2 => Re * Re + Im * Im;
    public bool EstNul => Re.EstNul && Im.EstNul;

    // Une partie imaginaire : i, -i, 2*i, (2/3)*i
    public static string EcritImaginaire(Rat im)
        => im.Num.Sign < 0 ? "-" + EcritImaginaire(-im)
         : im == Rat.One ? "i"
         : im.Den.IsOne ? im + "*i"
         : "(" + im + ")*i";

    public override string ToString()
    {
        if (Im.EstNul) return Re.ToString();
        if (Re.EstNul) return EcritImaginaire(Im);
        return "(" + Re + (Im.Num.Sign < 0 ? " - " : " + ") + EcritImaginaire(Im.Abs) + ")";
    }
}

// ---------- Monome : variables triees, exposants non nuls ----------
readonly struct Mon : IEquatable<Mon>
{
    public readonly int[] V;
    public readonly int[] E;

    public Mon(int[] v, int[] e) { V = v; E = e; }

    public static readonly Mon Constante = new Mon(Array.Empty<int>(), Array.Empty<int>());
    public static Mon De(int variable) => new Mon(new[] { variable }, new[] { 1 });

    // Degre total : somme des exposants.
    public int Degre
    {
        get
        {
            var d = 0;
            foreach (var e in E) d += e;
            return d;
        }
    }

    // Le monome dont l'exposant d'indice k est diminue de 1
    public Mon Diminue(int k)
    {
        if (E[k] > 1)
        {
            var e = (int[])E.Clone();
            e[k]--;
            return new Mon(V, e);
        }
        return Retiree(k);
    }

    // Le monome dont la variable d'indice k est retiree
    Mon Retiree(int k)
    {
        if (V.Length == 1) return Constante;
        var v = new int[V.Length - 1];
        var e = new int[E.Length - 1];
        Array.Copy(V, 0, v, 0, k);
        Array.Copy(V, k + 1, v, k, V.Length - k - 1);
        Array.Copy(E, 0, e, 0, k);
        Array.Copy(E, k + 1, e, k, E.Length - k - 1);
        return new Mon(v, e);
    }

    public static Mon operator *(Mon a, Mon b)
    {
        var v = new int[a.V.Length + b.V.Length];
        var e = new int[v.Length];
        int i = 0, j = 0, n = 0;
        while (i < a.V.Length && j < b.V.Length)
        {
            if (a.V[i] == b.V[j]) { v[n] = a.V[i]; e[n] = a.E[i] + b.E[j]; i++; j++; }
            else if (a.V[i] < b.V[j]) { v[n] = a.V[i]; e[n] = a.E[i]; i++; }
            else { v[n] = b.V[j]; e[n] = b.E[j]; j++; }
            n++;
        }
        while (i < a.V.Length) { v[n] = a.V[i]; e[n] = a.E[i]; i++; n++; }
        while (j < b.V.Length) { v[n] = b.V[j]; e[n] = b.E[j]; j++; n++; }
        return n == v.Length ? new Mon(v, e) : new Mon(v[..n], e[..n]);
    }

    public bool Equals(Mon autre)
    {
        if (V.Length != autre.V.Length) return false;
        for (int k = 0; k < V.Length; k++) if (V[k] != autre.V[k] || E[k] != autre.E[k]) return false;
        return true;
    }
    public override bool Equals(object objet) => objet is Mon autre && Equals(autre);
    public override int GetHashCode()
    {
        var h = new HashCode();
        for (int k = 0; k < V.Length; k++) { h.Add(V[k]); h.Add(E[k]); }
        return h.ToHashCode();
    }
}

// ---------- Polynome a coefficients complexes exacts ----------
// Les variables sont la coordonnee t, la coordonnee x, et des parametres
// constants (masse, charge, couplage) dont les derivees sont nulles.
sealed class Poly
{
    const int Constant = -1;

    static readonly Dictionary<int, int> roles = new();      // variable -> 0 pour t, 1 pour x, Constant pour un parametre
    static readonly Dictionary<int, string> noms = new();
    readonly Dictionary<Mon, Cpx> termes;                    // jamais de coefficient nul

    Poly(Dictionary<Mon, Cpx> termes) { this.termes = termes; }

    public static readonly Poly Zero = new Poly(new Dictionary<Mon, Cpx>());
    public static readonly Poly Un = Constante(Cpx.Un);

    public static Poly Constante(Cpx c)
    {
        if (c.EstNul) return Zero;
        return new Poly(new Dictionary<Mon, Cpx> { [Mon.Constante] = c });
    }
    public static implicit operator Poly(Cpx c) => Constante(c);
    public static implicit operator Poly(Rat r) => Constante(r);
    public static implicit operator Poly(long v) => Constante((Rat)v);

    // Une variable nommee : role 0 pour t, 1 pour x, Constant pour un parametre.
    public static Poly Variable(string nom, int role)
    {
        int id = noms.Count;
        noms[id] = nom;
        roles[id] = role;
        return new Poly(new Dictionary<Mon, Cpx> { [Mon.De(id)] = Cpx.Un });
    }

    // Un parametre constant (masse, charge, couplage) : ses derivees sont nulles.
    public static Poly Parametre(string nom) => Variable(nom, Constant);

    // Degre total maximum des monomes qu'un produit peut produire : -1 = aucun
    // plafond. Les developpements de Taylor tronques de la section 7 le reglent a 2.
    public static int Ordre = -1;

    static void Ajouter(Dictionary<Mon, Cpx> cible, Mon monome, Cpx coefficient)
    {
        if (coefficient.EstNul) return;
        if (Ordre >= 0 && monome.Degre > Ordre) return;
        if (cible.TryGetValue(monome, out var ancien))
        {
            var somme = ancien + coefficient;
            if (somme.EstNul) cible.Remove(monome); else cible[monome] = somme;
        }
        else cible[monome] = coefficient;
    }

    public bool EstNul => termes.Count == 0;
    public int NombreTermes => termes.Count;

    // Vrai si aucun terme ne porte ni variable ni parametre : un nombre pur.
    public bool EstNombre
    {
        get
        {
            foreach (var (m, _) in termes) if (m.V.Length > 0) return false;
            return true;
        }
    }

    public static Poly operator +(Poly a, Poly b)
    {
        var r = new Dictionary<Mon, Cpx>(a.termes);
        foreach (var (m, c) in b.termes) Ajouter(r, m, c);
        return new Poly(r);
    }
    public static Poly operator -(Poly a, Poly b) => a + (-b);
    public static Poly operator -(Poly a)
    {
        var r = new Dictionary<Mon, Cpx>();
        foreach (var (m, c) in a.termes) r[m] = -c;
        return new Poly(r);
    }
    public static Poly operator *(Poly a, Poly b)
    {
        if (a.EstNul || b.EstNul) return Zero;
        if (b.termes.Count < a.termes.Count) return b * a;
        var r = new Dictionary<Mon, Cpx>();
        foreach (var (ma, ca) in a.termes)
            foreach (var (mb, cb) in b.termes)
                Ajouter(r, ma * mb, ca * cb);
        return new Poly(r);
    }
    public static Poly operator *(Poly a, long n) => a * (Rat)n;

    // Division par un entier non nul : le coefficient devient rationnel.
    public static Poly operator /(Poly a, long n) => a * new Rat(1, n);

    public static bool operator ==(Poly a, Poly b)
    {
        if (ReferenceEquals(a, b)) return true;
        if (a is null || b is null || a.termes.Count != b.termes.Count) return false;
        foreach (var (m, c) in a.termes)
            if (!b.termes.TryGetValue(m, out var autre) || autre != c) return false;
        return true;
    }
    public static bool operator !=(Poly a, Poly b) => !(a == b);
    public override bool Equals(object objet) => objet is Poly autre && this == autre;
    public override int GetHashCode()
    {
        var h = 0;
        foreach (var c in termes.Values) h ^= c.GetHashCode();
        return h;
    }

    // Derivee partielle : direction 0 = t, 1 = x.
    Poly Derivee(int direction)
    {
        var r = new Dictionary<Mon, Cpx>();
        foreach (var (m, c) in termes)
            for (int k = 0; k < m.V.Length; k++)
            {
                if (roles[m.V[k]] != direction) continue;
                Ajouter(r, m.Diminue(k), c * m.E[k]);
            }
        return new Poly(r);
    }
    public Poly Dt => Derivee(0);
    public Poly Dx => Derivee(1);

    // Conjuguee : les variables sont reelles, seuls les coefficients changent.
    public Poly Conj
    {
        get
        {
            var r = new Dictionary<Mon, Cpx>();
            foreach (var (m, c) in termes) r[m] = c.Conj;
            return new Poly(r);
        }
    }

    // Plus grand rationnel g tel que tous les coefficients soient g fois un
    // entier gaussien ; 0 pour le polynome nul.
    public Rat Contenu
    {
        get
        {
            if (termes.Count == 0) return Rat.Zero;
            BigInteger pgcd = 0, ppmc = 1;
            foreach (var (_, c) in termes)
            {
                pgcd = BigInteger.GreatestCommonDivisor(pgcd, BigInteger.Abs(c.Re.Num));
                pgcd = BigInteger.GreatestCommonDivisor(pgcd, BigInteger.Abs(c.Im.Num));
                ppmc = ppmc * c.Re.Den / BigInteger.GreatestCommonDivisor(ppmc, c.Re.Den);
                ppmc = ppmc * c.Im.Den / BigInteger.GreatestCommonDivisor(ppmc, c.Im.Den);
            }
            return new Rat(pgcd, ppmc);
        }
    }

    // Le polynome divise par un rationnel non nul.
    public Poly Sur(Rat g)
    {
        var r = new Dictionary<Mon, Cpx>();
        foreach (var (m, c) in termes) r[m] = new Cpx(c.Re / g, c.Im / g);
        return new Poly(r);
    }

    // Valeur en un point : chaque variable (t, x, parametre) doit avoir la sienne.
    public Cpx En(IDictionary<string, Cpx> valeurs)
    {
        var somme = Cpx.Zero;
        foreach (var (m, c) in termes)
        {
            var facteur = Cpx.Un;
            for (int k = 0; k < m.V.Length; k++)
            {
                if (!valeurs.TryGetValue(noms[m.V[k]], out var valeur))
                    throw new InvalidOperationException("valeur manquante pour " + noms[m.V[k]]);
                for (int e = 0; e < m.E[k]; e++) facteur = facteur * valeur;
            }
            somme = somme + c * facteur;
        }
        return somme;
    }

    static string Nom(int variable, int exposant)
        => exposant == 1 ? noms[variable] : noms[variable] + "^" + exposant;

    static string EcritCoef(Cpx c)
    {
        if (c == Cpx.Un) return "";
        if (c == -Cpx.Un) return "-";
        if (c.Im.EstNul) return Rat.Ecrit(c.Re);
        if (c.Re.EstNul) return Cpx.EcritImaginaire(c.Im);
        return c.ToString();
    }

    public override string ToString()
    {
        if (termes.Count == 0) return "0";
        var lignes = new List<(int degre, string texte, string cle)>();
        foreach (var (m, c) in termes)
        {
            var produit = string.Join("*", m.V.Zip(m.E, Nom));
            var coef = EcritCoef(c);
            var texte = m.V.Length == 0 ? c.ToString()
                                     : coef == "" ? produit : coef == "-" ? "-" + produit : coef + "*" + produit;
            lignes.Add((m.E.Sum(), texte, produit));
        }
        return string.Join(" + ", lignes.OrderBy(z => z.degre).ThenBy(z => z.cle).Select(z => z.texte))
                     .Replace("+ -", "- ");
    }
}

// ---------- Fraction rationnelle de polynomes ----------
sealed class Frac
{
    public Poly Num { get; }
    public Poly Den { get; }

    public static readonly Frac Zero = new Frac(Poly.Zero, Poly.Un);

    public Frac(Poly num, Poly den)
    {
        if (den.EstNul) throw new DivideByZeroException("fraction a denominateur nul");
        // Le contenu rationnel commun au numerateur et au denominateur est retire.
        var contenuNum = num.Contenu;
        var contenuDen = den.Contenu;
        var commun = new Rat(BigInteger.GreatestCommonDivisor(contenuNum.Num, contenuDen.Num),
                             contenuNum.Den * contenuDen.Den
                                 / BigInteger.GreatestCommonDivisor(contenuNum.Den, contenuDen.Den));
        Num = commun == Rat.One ? num : num.Sur(commun);
        Den = commun == Rat.One ? den : den.Sur(commun);
    }

    public static implicit operator Frac(Poly p) => new Frac(p, Poly.Un);
    public static implicit operator Frac(Cpx c) => new Frac(c, Poly.Un);
    public static implicit operator Frac(Rat r) => new Frac(r, Poly.Un);
    public static implicit operator Frac(long v) => new Frac(v, Poly.Un);

    public static Frac operator +(Frac a, Frac b) => new Frac(a.Num * b.Den + b.Num * a.Den, a.Den * b.Den);
    public static Frac operator -(Frac a, Frac b) => new Frac(a.Num * b.Den - b.Num * a.Den, a.Den * b.Den);
    public static Frac operator -(Frac a) => new Frac(-a.Num, a.Den);
    public static Frac operator *(Frac a, Frac b) => new Frac(a.Num * b.Num, a.Den * b.Den);
    public static Frac operator /(Frac a, Frac b)
        => b.Num.EstNul ? throw new DivideByZeroException("fraction a numerateur nul au denominateur")
                        : new Frac(a.Num * b.Den, a.Den * b.Num);

    public Frac Dt => new Frac(Num.Dt * Den - Num * Den.Dt, Den * Den);
    public Frac Dx => new Frac(Num.Dx * Den - Num * Den.Dx, Den * Den);
    public Frac Conj => new Frac(Num.Conj, Den.Conj);

    public bool EstNulle => Num.EstNul;
    public Cpx En(IDictionary<string, Cpx> valeurs) => Num.En(valeurs) / Den.En(valeurs);

    // Un denominateur s'ecrit sans parentheses seulement s'il se reduit a un
    // produit de variables (t, x^2) : "t / 5*x" se lirait a tort "t/5 fois x".
    static string EcritDen(Poly p)
    {
        var texte = p.ToString();
        var nu = p.NombreTermes == 1 && texte[0] >= 'a' && texte[0] <= 'z';
        return nu || p.EstNombre ? texte : "(" + texte + ")";
    }

    public override string ToString()
        => Num.EstNul ? "0" : Den == Poly.Un ? Num.ToString()
                     : (Num.NombreTermes <= 1 ? Num.ToString() : "(" + Num + ")") + " / " + EcritDen(Den);
}

// ---------- Matrices 2x2 : tableaux imbriques de Frac, et spineurs ----------
static class M2
{
    public static readonly Frac[][] Identite = { new Frac[] { 1, 0 }, new Frac[] { 0, 1 } };

    public static Frac[][] Matrice(Frac a, Frac b, Frac c, Frac d)
        => new Frac[][] { new Frac[] { a, b }, new Frac[] { c, d } };

    public static Frac[][] Mul(Frac[][] a, Frac[][] b)
        => Matrice(a[0][0] * b[0][0] + a[0][1] * b[1][0], a[0][0] * b[0][1] + a[0][1] * b[1][1],
                   a[1][0] * b[0][0] + a[1][1] * b[1][0], a[1][0] * b[0][1] + a[1][1] * b[1][1]);

    public static Frac[] MulV(Frac[][] a, Frac[] v)
        => new[] { a[0][0] * v[0] + a[0][1] * v[1], a[1][0] * v[0] + a[1][1] * v[1] };

    public static Frac[][] Add(Frac[][] a, Frac[][] b)
        => Matrice(a[0][0] + b[0][0], a[0][1] + b[0][1], a[1][0] + b[1][0], a[1][1] + b[1][1]);

    public static Frac[][] Sub(Frac[][] a, Frac[][] b)
        => Matrice(a[0][0] - b[0][0], a[0][1] - b[0][1], a[1][0] - b[1][0], a[1][1] - b[1][1]);

    public static Frac[] Sub(Frac[] a, Frac[] b) => new[] { a[0] - b[0], a[1] - b[1] };

    public static Frac[][] Echelle(Frac k, Frac[][] a)
        => Matrice(k * a[0][0], k * a[0][1], k * a[1][0], k * a[1][1]);

    // Transposee conjuguee : la transposee seule ne suffit pas (sigma_2 est antisymetrique).
    public static Frac[][] Dag(Frac[][] a)
        => Matrice(a[0][0].Conj, a[1][0].Conj, a[0][1].Conj, a[1][1].Conj);

    public static Frac[][] DeriveeT(Frac[][] a)
        => Matrice(a[0][0].Dt, a[0][1].Dt, a[1][0].Dt, a[1][1].Dt);

    public static Frac[][] DeriveeX(Frac[][] a)
        => Matrice(a[0][0].Dx, a[0][1].Dx, a[1][0].Dx, a[1][1].Dx);

    public static bool EstNulle(Frac[][] a)
        => a[0][0].EstNulle && a[0][1].EstNulle && a[1][0].EstNulle && a[1][1].EstNulle;

    public static bool EstNul(Frac[] v) => v[0].EstNulle && v[1].EstNulle;

    public static Frac Trace(Frac[][] a) => a[0][0] + a[1][1];
}

// ---------- Matrices evaluees en un point : tableaux imbriques de Cpx ----------
static class Pt
{
    public static readonly Cpx[][] Identite = { new Cpx[] { 1, 0 }, new Cpx[] { 0, 1 } };

    public static Cpx[][] Matrice(Cpx a, Cpx b, Cpx c, Cpx d)
        => new Cpx[][] { new Cpx[] { a, b }, new Cpx[] { c, d } };

    public static Cpx[][] Evalue(Frac[][] a, IDictionary<string, Cpx> valeurs)
        => Matrice(a[0][0].En(valeurs), a[0][1].En(valeurs), a[1][0].En(valeurs), a[1][1].En(valeurs));

    public static Cpx[][] Mul(Cpx[][] a, Cpx[][] b)
        => Matrice(a[0][0] * b[0][0] + a[0][1] * b[1][0], a[0][0] * b[0][1] + a[0][1] * b[1][1],
                   a[1][0] * b[0][0] + a[1][1] * b[1][0], a[1][0] * b[0][1] + a[1][1] * b[1][1]);

    public static Cpx[][] Dag(Cpx[][] a)
        => Matrice(a[0][0].Conj, a[1][0].Conj, a[0][1].Conj, a[1][1].Conj);

    // Plus grand module des ecarts de deux matrices, arrondi a huit decimales.
    public static Rat EcartMax(Cpx[][] a, Cpx[][] b)
    {
        var max = Rat.Zero;
        for (int i = 0; i < 2; i++)
            for (int j = 0; j < 2; j++)
            {
                var e = (a[i][j] - b[i][j]).Norme2.Racine();
                if (e > max) max = e;
            }
        return max;
    }
}

// ---------- Assertion executable : leve une exception si la condition echoue ----------
void Exige(bool condition, string libelle)
{
    if (!condition) throw new InvalidOperationException("Assertion echouee : " + libelle);
    Console.WriteLine("OK   " + libelle);
}

// ---------- Symbole de Levi-Civita, indices 0, 1, 2 ----------
int Eps(int a, int b, int c)
{
    if (a == b || b == c || a == c) return 0;
    return (a < b && b < c) || (b < c && c < a) || (c < a && a < b) ? 1 : -1;
}

// ==========================================================================
// Section 1 -- symetrie U(1) globale d'un champ scalaire complexe
// ==========================================================================

// Les coordonnees (t, x), et les parametres constants : masse, charge, couplage.
Poly t = Poly.Variable("t", 0);
Poly x = Poly.Variable("x", 1);
var m = Poly.Parametre("m");
var q = Poly.Parametre("q");
var g = Poly.Parametre("g");

// Champs scalaires reels de (t, x). La source les laisse non specifiees
// (calcul symbolique) ; ici ce sont des polynomes, et chaque identite est
// verifiee comme identite EXACTE entre fonctions rationnelles de (t, x).
Poly u = new Rat(1, 2) + t - x / 3 + t * x / 4;
Poly v = t / 4 + x / 5 - t * x / 6 + x * x / 7;
Poly A_t = t / 5 - x / 2 + t * x / 4;
Poly A_x = x / 3 + t / 4 - t * x / 5;

// Lagrangien du champ libre (phi = u + i v, sans champ de jauge) :
// L = |d_t phi|^2 - |d_x phi|^2 - m^2 |phi|^2.
Frac L_libre(Frac ph, Poly masse)
    => ph.Dt.Conj * ph.Dt - ph.Dx.Conj * ph.Dx - masse * masse * ph.Conj * ph;

// Une phase e^(i alpha), alpha constante ou fonction de (t, x), se decrit par
// s = tan(alpha/2) : e^(i alpha) = (1 + i s) / (1 - i s). Tout reste rationnel.
Frac EPhase(Poly s) => new Frac(1 + Cpx.I * s, 1 - Cpx.I * s);

// Derivee de la phase : d alpha = 2 d s / (1 + s^2).
Frac DAlpha(Poly s, Poly derivee) => new Frac(2 * derivee, 1 + s * s);

// Derivee covariante D_mu phi = d_mu phi + i q A_mu phi.
Frac DCov(Frac ph, Frac dphi, Frac A, Poly charge) => dphi + Cpx.I * charge * A * ph;

var phi = u + Cpx.I * v;

// Transformation de phase globale : alpha constant.
var tan_demi_alpha = Poly.Parametre("a");
var delta_global = L_libre(EPhase(tan_demi_alpha) * phi, m) - L_libre(phi, m);

Console.WriteLine("delta L (phase globale, alpha constant) = " + delta_global);
Exige(delta_global.EstNulle,
      "phase globale : delta L = 0, identite exacte pour ces champs et toute phase constante");

delta L (phase globale, alpha constant) = 0


OK   phase globale : delta L = 0, identite exacte pour ces champs et toute phase constante


### Lecture du résultat

La phase globale laisse le lagrangien **inchangé** : `delta L (phase globale, alpha constant) = 0`. Ce zéro est un zéro **exact** : le reste calculé est le polynôme nul, et non un nombre plus petit qu'un seuil. La raison tient en une ligne : sous $\phi \to e^{i\alpha}\phi$ avec $\alpha$ **constant**, le terme de masse $m^2\phi^*\phi$ est inchangé parce que $e^{i\alpha}e^{-i\alpha} = 1$, et le terme cinétique $\partial_\mu\phi^*\partial^\mu\phi$ l'est aussi parce que la dérivée d'une phase constante est nulle. C'est la symétrie $U(1)$ **globale** du champ scalaire complexe : elle ne fait intervenir aucun champ auxiliaire, le lagrangien ne dépend pas du tout de $\alpha$, et le courant conservé associé — la charge $U(1)$ — est celui que le carnet Jauge-01 a établi.

## 2. Le problème : rendre la phase locale

Rendons maintenant $\alpha$ une **fonction** $\alpha(x,t)$ : chaque point de l'espace-temps choisit sa propre phase. C'est l'exigence de Yang–Mills transposée au champ chargé (« the arbitrariness in choosing the phase factor is local in character », copie archivée p. 192).

In [2]:
// Phase locale : alpha devient une fonction alpha(t, x)
// Elle est decrite par son demi-angle tangente s = tan(alpha/2), polynome en (t, x).
Poly s = t / 3 + x * x / 5 - t * x / 7;
var delta_local = L_libre(EPhase(s) * phi, m) - L_libre(phi, m);

// Forme fermee de l'ecart, avec J_mu = u d_mu v - v d_mu u :
// delta L = |phi|^2 [(d_t alpha)^2 - (d_x alpha)^2] + 2 [(d_t alpha) J_t - (d_x alpha) J_x].
var alpha_t = DAlpha(s, s.Dt);
var alpha_x = DAlpha(s, s.Dx);
var module2 = phi.Conj * phi;
var J_t = u * v.Dt - v * u.Dt;
var J_x = u * v.Dx - v * u.Dx;
var forme_fermee = module2 * (alpha_t * alpha_t - alpha_x * alpha_x)
    + 2 * (alpha_t * J_t - alpha_x * J_x);

// On isole les termes de derivee de alpha (ils ne se compensent pas).
Console.WriteLine("delta L (phase locale) -- expression complete, facteurs en d(alpha) :");
Console.WriteLine("   |phi|^2 [(d_t alpha)^2 - (d_x alpha)^2] + 2 [(d_t alpha)(u d_t v - v d_t u)"
    + " - (d_x alpha)(u d_x v - v d_x u)]");
Exige((delta_local - forme_fermee).EstNulle,
    "phase locale : delta L = |phi|^2 [(d_t alpha)^2 - (d_x alpha)^2] + 2 (d alpha)(u dv - v du), identite exacte");
Exige(!delta_local.EstNulle,
    "phase locale : delta L n'est pas nul -- l'invariance est brisee");

// Complement de la mesure : on neutralise les derivees de alpha, ce qui
// ramene la phase LOCALE a la phase GLOBALE -- et le Lagrangien doit alors
// etre invariant. Toute la rupture vient de d(alpha), rien d'autre.
var tan_demi_alpha_neutre = Poly.Parametre("s0");
var delta_sans_derivees = L_libre(EPhase(tan_demi_alpha_neutre) * phi, m) - L_libre(phi, m);
Console.WriteLine("delta L (d(alpha) neutralise -- phase globale) = " + delta_sans_derivees);
Exige(delta_sans_derivees.EstNulle,
    "phase locale, d(alpha) neutralise : delta L = 0 -- la rupture vient de d(alpha), et de rien d'autre");

delta L (phase locale) -- expression complete, facteurs en d(alpha) :


   |phi|^2 [(d_t alpha)^2 - (d_x alpha)^2] + 2 [(d_t alpha)(u d_t v - v d_t u) - (d_x alpha)(u d_x v - v d_x u)]


OK   phase locale : delta L = |phi|^2 [(d_t alpha)^2 - (d_x alpha)^2] + 2 (d alpha)(u dv - v du), identite exacte


OK   phase locale : delta L n'est pas nul -- l'invariance est brisee


delta L (d(alpha) neutralise -- phase globale) = 0


OK   phase locale, d(alpha) neutralise : delta L = 0 -- la rupture vient de d(alpha), et de rien d'autre


### Lecture du résultat

La phase locale produit une expression **non nulle**, et sa structure le dit précisément : elle ne dépend que des **dérivées** de $\alpha$, jamais de $\alpha$ lui-même. La cellule ne se contente pas de l'affirmer : elle établit que l'écart calculé est **identique** à la forme fermée

$$\delta L = |\varphi|^2\left[(\partial_t\alpha)^2 - (\partial_x\alpha)^2\right] + 2\left[(\partial_t\alpha)\,J_t - (\partial_x\alpha)\,J_x\right], \qquad J_\mu = u\,\partial_\mu v - v\,\partial_\mu u ,$$

puis que cet écart n'est pas le polynôme nul.

Le coefficient de $(\partial_t\alpha)^2$ vaut $+|\varphi|^2$ et celui de $(\partial_x\alpha)^2$ vaut $-|\varphi|^2$ — c'est la signature $(+,-)$ qui décide du signe. Les termes croisés associent $\partial_t\alpha$ à $u\,\partial_t v - v\,\partial_t u$ et $\partial_x\alpha$ à $u\,\partial_x v - v\,\partial_x u$, avec le signe opposé : la partie réelle et la partie imaginaire ne se transforment pas indépendamment.

Le test décisif est la dernière ligne : une fois les dérivées de $\alpha$ neutralisées — ce qui ramène la phase locale à la phase globale — l'écart redevient **exactement 0** (`delta L (d(alpha) neutralise -- phase globale) = 0`). Toute la rupture vient donc de $\partial_\mu\alpha$, et de rien d'autre. Les termes en $\partial_\mu\alpha$ **ne se compensent pas** : la symétrie globale ne survit pas à la localisation. C'est le point de départ historique de toute l'électrodynamique de jauge.

## 3. Dérivée covariante et champ de jauge

La réparation : remplacer $\partial_\mu$ par la **dérivée covariante**

$$D_\mu = \partial_\mu + iq A_\mu, \qquad A_\mu \to A_\mu - \frac{1}{q}\partial_\mu \alpha,$$

et reconstruire $\mathcal{L}_{\text{jauge}} = (D_\mu\varphi)^* (D^\mu\varphi) - m^2\varphi^*\varphi$. Vérifions d'abord **exactement** que $D_\mu\varphi$ se transforme comme $\varphi$ lui-même.

In [3]:
// Propriete de covariance de la derivee covariante (exacte)

// Cote transforme : phi' = exp(i*alpha)*phi, A'_mu = A_mu - d_mu(alpha)/q.
var e_i = EPhase(s);
var A_t_p = A_t - alpha_t / q;
var A_x_p = A_x - alpha_x / q;
var phi_p = e_i * phi;

var Dphi_t = DCov(phi, phi.Dt, A_t, q);
var Dphi_x = DCov(phi, phi.Dx, A_x, q);
var Dphi_t_p = DCov(phi_p, phi_p.Dt, A_t_p, q);
var Dphi_x_p = DCov(phi_p, phi_p.Dx, A_x_p, q);

var cov_t = Dphi_t_p - e_i * Dphi_t;
var cov_x = Dphi_x_p - e_i * Dphi_x;
Console.WriteLine("D'_t(phi') - e^(i alpha) D_t(phi) = " + cov_t);
Console.WriteLine("D'_x(phi') - e^(i alpha) D_x(phi) = " + cov_x);
Exige(cov_t.EstNulle && cov_x.EstNulle,
    "covariance : D'_mu phi' = e^(i alpha) D_mu phi, en t et en x (identite exacte)");

D'_t(phi') - e^(i alpha) D_t(phi) = 0


D'_x(phi') - e^(i alpha) D_x(phi) = 0


OK   covariance : D'_mu phi' = e^(i alpha) D_mu phi, en t et en x (identite exacte)


**De la covariance à l'invariance.** La covariance — $D_\mu\varphi$ se transforme comme $\varphi$ lui-même — est une condition **nécessaire**, pas suffisante : elle dit que le terme de couplage se comporte correctement, pas que le lagrangien est invariant. Passons à la vérification **complète** : le lagrangien couplé, masse comprise, doit être inchangé par la transformation de jauge.

In [4]:
// Invariance de jauge EXACTE du Lagrangien couple.
// La transformation est ecrite en forme COMPLEXE (exp(i alpha)) plutot qu'en
// composantes reelles (cos, sin) : l'algebre reste dans les fractions
// rationnelles, et le resultat attendu est exactement nul.

// D_mu ph = d_mu ph + i q A_mu ph ; L = |D_t ph|^2 - |D_x ph|^2 - m^2 |ph|^2
Frac lagrange_jauge(Frac At, Frac Ax, Frac ph)
{
    var Dt = DCov(ph, ph.Dt, At, q);
    var Dx = DCov(ph, ph.Dx, Ax, q);
    return Dt.Conj * Dt - Dx.Conj * Dx - m * m * ph.Conj * ph;
}

var L_couplage = lagrange_jauge(A_t, A_x, phi);
var L_couplage_p = lagrange_jauge(A_t_p, A_x_p, e_i * phi);
var delta_jauge = L_couplage_p - L_couplage;
Console.WriteLine("delta L_jauge (transformation locale complete) = " + delta_jauge);
Exige(delta_jauge.EstNulle,
    "invariance de jauge : delta L_jauge = 0 sous la transformation locale complete (identite exacte)");

delta L_jauge (transformation locale complete) = 0


OK   invariance de jauge : delta L_jauge = 0 sous la transformation locale complete (identite exacte)


### Lecture du résultat

L'invariance de jauge est **exacte**, pas approchée : `delta L_jauge (transformation locale complete) = 0`. Deux effets se compensent exactement. D'un côté, la dérivée covariante gagne $\pm iqA_\mu\phi$ ; de l'autre, le potentiel de jauge se transforme en $A_\mu \to A_\mu - \tfrac1q\partial_\mu\alpha$, et la dérivée de $\alpha$ qui apparaît ainsi compense le terme précédent. Ni la masse, ni le terme cinétique ne laissent de résidu.

C'est là que se joue la différence avec la symétrie globale : pour la phase globale, le lagrangien ne dépendait **pas du tout** de $\alpha$ ; pour la phase locale, il ne dépend plus que de $\partial_\mu\alpha$ — et cette dépendance a été absorbée par le champ auxiliaire $A_\mu$, qui devient une partie prenante du modèle et non un artifice de calcul.

Remarque de méthode : la phase est écrite sous sa forme complexe $(1+is)/(1-is)$, et non en composantes réelles (cosinus, sinus). L'algèbre reste ainsi dans les fractions rationnelles, et le zéro affiché est bien le polynôme nul.

## 4. Le tenseur $F_{\mu\nu}$ et le Lagrangien complet

Le champ $A_\mu$ a lui-même une dynamique. Son tenseur de courbure abélien $F_{\mu\nu} = \partial_\mu A_\nu - \partial_\nu A_\mu$ est **invariant** sous la jauge ; le terme cinétique $-\tfrac14 F_{\mu\nu}F^{\mu\nu}$ complète le Lagrangien scalaire de l'électrodynamique (sqED en 1+1 dimensions).

In [5]:
// F est invariant sous la transformation de jauge abelienne
Frac F_tx = A_x.Dt - A_t.Dx;
var F_tx_p = A_x_p.Dt - A_t_p.Dx;
var ecart_F = F_tx_p - F_tx;
Console.WriteLine("F' - F = " + ecart_F);
Exige(ecart_F.EstNulle, "F_tx = d_t A_x - d_x A_t est invariant : F' - F = 0 (identite exacte)");

// Lagrangien complet : champ couple + terme cinetique du champ de jauge.
// En 1+1 D, -1/4 F_mu nu F^mu nu = +F_tx^2 / 2 (metrique +,-)
var L_F = F_tx * F_tx / 2;
var L_total = L_couplage + L_F;
var L_total_p = lagrange_jauge(A_t_p, A_x_p, e_i * phi) + F_tx_p * F_tx_p / 2;
var delta_total = L_total_p - L_total;
Console.WriteLine("delta L_total (jauge locale) = " + delta_total);
Exige(delta_total.EstNulle,
    "invariance de jauge du Lagrangien total : delta L_total = 0 (identite exacte)");

F' - F = 0


OK   F_tx = d_t A_x - d_x A_t est invariant : F' - F = 0 (identite exacte)


delta L_total (jauge locale) = 0


OK   invariance de jauge du Lagrangien total : delta L_total = 0 (identite exacte)


### Lecture du résultat

Le champ $F_{\mu\nu}$ est **invariant** sous la jauge : `F' - F = 0`, exactement. La raison tient en une ligne — dans $\partial_\mu A'_\nu - \partial_\nu A'_\mu$, les deux dérivées secondes de $\alpha$ apportées par la transformation, $-\tfrac1q\partial_\mu\partial_\nu\alpha$ et $-\tfrac1q\partial_\nu\partial_\mu\alpha$, se retranchent. Il ne reste que $F_{\mu\nu}$ lui-même.

Et surtout, le **Lagrangien complet** est lui aussi invariant : `delta L_total (jauge locale) = 0`, une fois le terme cinétique $-\tfrac14 F_{\mu\nu}F^{\mu\nu}$ ajouté au lagrangien couplé. C'est le résultat qui compte physiquement : on n'a pas introduit un champ auxiliaire gratuit, on a introduit un champ dont la **propre** dynamique est invariante de jauge. En $1+1$ dimension et pour la signature $(+,-)$, ce terme cinétique vaut $\tfrac12 F_{tx}^2$.

## 5. Structure SU(2) : matrices de Pauli et constantes de structure

Chez Yang–Mills, le groupe n'est plus le cercle U(1) mais **SU(2)** (le spin isotopique du papier de 1954). Les générateurs $T^a = \sigma^a/2$ ($\sigma^a$ matrices de Pauli) obéissent à

$$[T^a, T^b] = i\,\varepsilon^{abc}\, T^c,$$

les $\varepsilon^{abc}$ sont les **constantes de structure** — c'est cette algèbre non abélienne qui va fabriquer l'auto-interaction.

En C#, une matrice est un tableau imbriqué de `Frac` : `sigma_1`, `sigma_2`, `sigma_3` sont les matrices de Pauli à coefficients rationnels exacts, et les neuf commutateurs sont comparés un à un à la formule.

In [6]:
// Verification exhaustive des 9 commutateurs [sigma_a/2, sigma_b/2]
// Matrices de Pauli : tableaux imbriques de rationnels exacts.
var sigma_1 = new Frac[][] { new Frac[] { 0, 1 }, new Frac[] { 1, 0 } };
var sigma_2 = new Frac[][] { new Frac[] { 0, -Cpx.I }, new Frac[] { Cpx.I, 0 } };
var sigma_3 = new Frac[][] { new Frac[] { 1, 0 }, new Frac[] { 0, -1 } };

// Les generateurs T^a = sigma_a / 2.
var demi = new Frac[][][]
{
    M2.Echelle(new Rat(1, 2), sigma_1),
    M2.Echelle(new Rat(1, 2), sigma_2),
    M2.Echelle(new Rat(1, 2), sigma_3),
};

// Combinaison lineaire de matrices : c_0 M_0 + c_1 M_1 + ...
Frac[][] Combinaison(Frac[] coefficients, Frac[][][] matrices)
{
    var resultat = M2.Echelle(0, M2.Identite);
    for (int i = 0; i < coefficients.Length; i++)
        resultat = M2.Add(resultat, M2.Echelle(coefficients[i], matrices[i]));
    return resultat;
}

// [T^a, T^b] = i eps_abc T^c, avec eps le symbole de Levi-Civita.
var paires = 0;
for (int i = 0; i < 3; i++)
    for (int j = 0; j < 3; j++)
    {
        var comm = M2.Sub(M2.Mul(demi[i], demi[j]), M2.Mul(demi[j], demi[i]));
        var attendu = Combinaison(new Frac[] { Cpx.I * Eps(i, j, 0), Cpx.I * Eps(i, j, 1), Cpx.I * Eps(i, j, 2) }, demi);
        if (M2.EstNulle(M2.Sub(comm, attendu))) paires++;
    }

Console.WriteLine("[sigma_a/2, sigma_b/2] = i eps_abc sigma_c/2 : " + paires + "/9 paires verifiees");
Exige(paires == 9, "algebre su(2) : les 9 commutateurs [sigma_a/2, sigma_b/2] valent i eps_abc sigma_c/2");

[sigma_a/2, sigma_b/2] = i eps_abc sigma_c/2 : 9/9 paires verifiees


OK   algebre su(2) : les 9 commutateurs [sigma_a/2, sigma_b/2] valent i eps_abc sigma_c/2


## 6. La courbure non abélienne : le commutateur $[D_\mu, D_\nu]$

En SU(2), la dérivée covariante devient **matricielle** : $D_\mu = \partial_\mu + ig W^a_\mu T^a$ où les trois champs $W^a_\mu$ ($a = 1,2,3$) forment un vecteur isotopique. Le tenseur de courbure se définit par le commutateur :

$$[D_\mu, D_\nu] = ig\, F^a_{\mu\nu}\, T^a, \qquad
F^a_{\mu\nu} = \partial_\mu W^a_\nu - \partial_\nu W^a_\mu \;\underline{- g\, \varepsilon^{abc} W^b_\mu W^c_\nu}.$$

**C'est le cœur de Yang–Mills 1954** : dans la copie archivée (p. 193), le tenseur du papier, noté $f_{\mu\nu} = \partial b_\mu/\partial x_\nu - \partial b_\nu/\partial x_\mu + 2g\, b_\mu \times b_\nu$ (en notant $g$ la constante de couplage du papier), porte le même terme bilinéaire $b_\mu \times b_\nu = \varepsilon^{abc} b^b b^c$ — le champ de jauge **s'auto-interagit**, même sans matière.

Les deux écritures n'ont pas le même signe, et c'est normal : la copie archivée remplace le gradient par $(\partial_\mu - 2ig\, b_\mu \cdot T)$, couplage de signe opposé au nôtre, là où ce carnet fixe $D_\mu = \partial_\mu + ig\,W^a_\mu T^a$ avec $T^a = \sigma^a/2$. Le signe du terme bilinéaire suit la convention de couplage, pas la physique. Plutôt que de le supposer, on le **vérifie** : le commutateur calculé plus bas doit reproduire la formule écrite, terme bilinéaire compris.

In [7]:
// Matrice de jauge W^a_mu T^a = W^a_mu sigma_a/2, spineur Psi, et commutateur
// [D_t, D_x] = i g F^a T^a : le coeur non abelien de la theorie.

// Composantes du champ de jauge W^a_mu et spineur Psi a deux composantes :
// polynomes en (t, x), comme les champs du debut du carnet. Les champs W servent
// aussi au controle de la section 7.
var W_t = new Frac[] { t / 5, new Rat(1, 5) - t * x / 4, x / 5 - t * t / 7 };
var W_x = new Frac[] { x / 4 - t / 3, t / 3 + x * x / 5, new Rat(1, 4) - t * x / 6 };
var Psi = new Frac[] { new Rat(1, 3) + t / 2 - x / 4, t / 5 + x / 3 + t * x / 4 };

// Matrice 2x2  W^a_mu T^a = W^a_mu sigma_a / 2
Frac[][] Wmat(Frac[] W_mu) => Combinaison(W_mu, demi);

// D_mu Psi = d_mu Psi + i g W^a_mu T^a Psi
Frac[] D(Frac[] vecteur, Frac[] W_mu, int mu)
{
    var W_Psi = M2.MulV(Wmat(W_mu), vecteur);
    var derivee = mu == 0 ? new[] { vecteur[0].Dt, vecteur[1].Dt } : new[] { vecteur[0].Dx, vecteur[1].Dx };
    return new[] { derivee[0] + Cpx.I * g * W_Psi[0], derivee[1] + Cpx.I * g * W_Psi[1] };
}

// [D_t, D_x] = D_t D_x - D_x D_t. L'ORDRE compte : l'ordre inverse
// calculerait [D_x, D_t], l'oppose, et le reste serait non nul alors que la
// formule de F est correcte.
Frac[] Commutateur(Frac[] Wt, Frac[] Wx, Frac[] spineur)
    => M2.Sub(D(D(spineur, Wx, 1), Wt, 0), D(D(spineur, Wt, 0), Wx, 1));

// Terme d'auto-interaction de F^a : -g eps_abc W^b_t W^c_x
Frac AutoInteraction(Frac[] Wt, Frac[] Wx, Poly couplage, int i)
{
    var terme = Frac.Zero;
    for (int j = 0; j < 3; j++)
        for (int l = 0; l < 3; l++)
            if (Eps(i, j, l) != 0) terme = terme - couplage * Eps(i, j, l) * Wt[j] * Wx[l];
    return terme;
}

// F^a_tx = d_t W^a_x - d_x W^a_t - g eps_abc W^b_t W^c_x
Frac[] ChampF(Frac[] Wt, Frac[] Wx, Poly couplage)
{
    var F = new Frac[3];
    for (int i = 0; i < 3; i++) F[i] = Wx[i].Dt - Wt[i].Dx + AutoInteraction(Wt, Wx, couplage, i);
    return F;
}

// Le reste  [D_t, D_x] Psi - i g F^a T^a Psi  pour un triplet de champs.
Frac[] Reste(Frac[] Wt, Frac[] Wx, Frac[] spineur)
{
    var f = ChampF(Wt, Wx, g);
    var attendu = M2.MulV(Combinaison(f, demi), spineur);
    return M2.Sub(Commutateur(Wt, Wx, spineur),
                  new[] { Cpx.I * g * attendu[0], Cpx.I * g * attendu[1] });
}

// Verdict sur les champs du carnet : le reste doit etre nul.
var reste = Reste(W_t, W_x, Psi);
Console.WriteLine("[D_t, D_x] - i g F^a sigma_a/2 applique a Psi (doit etre nul) :");
Console.WriteLine("  composante 1 : " + reste[0]);
Console.WriteLine("  composante 2 : " + reste[1]);
Exige(M2.EstNul(reste), "non abelien : [D_t, D_x] Psi = i g F^a T^a Psi (identite exacte)");

// Meme verification sur des champs a coefficients symboliques : chaque champ
// est a + b t + c x + d t x, ou a, b, c, d sont des parametres libres (32 en
// tout, le couplage g a part). L'identite ne doit rien a un choix de champs :
// elle vaut pour toute valeur de ces parametres.
Poly ChampGenerique(string nom)
    => Poly.Parametre(nom + "0") + Poly.Parametre(nom + "1") * t
       + Poly.Parametre(nom + "2") * x + Poly.Parametre(nom + "3") * t * x;
var W_t_gen = new Frac[] { ChampGenerique("W1t"), ChampGenerique("W2t"), ChampGenerique("W3t") };
var W_x_gen = new Frac[] { ChampGenerique("W1x"), ChampGenerique("W2x"), ChampGenerique("W3x") };
var Psi_gen = new Frac[] { ChampGenerique("psi1"), ChampGenerique("psi2") };
var reste_gen = Reste(W_t_gen, W_x_gen, Psi_gen);
Console.WriteLine("Champs a coefficients symboliques (32 parametres libres) : reste nul = " + M2.EstNul(reste_gen));
Exige(M2.EstNul(reste_gen),
    "non abelien, champs a coefficients symboliques : meme identite pour toute valeur des 32 parametres");

// Le terme d'auto-interaction, d'abord sur des etiquettes algebriques (pour
// lire sa structure), puis sur les champs W du carnet.
var etiquette_t = new Frac[] { Poly.Parametre("W1t"), Poly.Parametre("W2t"), Poly.Parametre("W3t") };
var etiquette_x = new Frac[] { Poly.Parametre("W1x"), Poly.Parametre("W2x"), Poly.Parametre("W3x") };
Console.WriteLine("Terme d'auto-interaction extrait de F^1 : " + AutoInteraction(etiquette_t, etiquette_x, g, 0));
Console.WriteLine("Valeur pour les champs W du carnet   : " + AutoInteraction(W_t, W_x, g, 0));

[D_t, D_x] - i g F^a sigma_a/2 applique a Psi (doit etre nul) :


  composante 1 : 0


  composante 2 : 0


OK   non abelien : [D_t, D_x] Psi = i g F^a T^a Psi (identite exacte)


Champs a coefficients symboliques (32 parametres libres) : reste nul = True


OK   non abelien, champs a coefficients symboliques : meme identite pour toute valeur des 32 parametres


Terme d'auto-interaction extrait de F^1 : -g*W2t*W3x + g*W3t*W2x


Valeur pour les champs W du carnet   : (-420*g + 1365*t*x*g - 400*t^3*g + 336*x^3*g - 590*t^2*x^2*g) / 8400


### Lecture du résultat

Le reste est **nul sur les deux composantes** : `composante 1 : 0`, `composante 2 : 0`. Le commutateur $[D_t,D_x]$ calculé à partir de la seule définition $D_\mu = \partial_\mu + igW^a_\mu T^a$ reproduit donc exactement $igF^a_{\mu\nu}T^a$, terme bilinéaire compris. La formule de la section 6 n'est pas une hypothèse que l'on vérifie : elle **tombe** de l'algèbre.

Ce premier calcul porte sur des champs choisis. Pour qu'il ne doive rien à ce choix, il est recommencé sur des champs à **coefficients symboliques** : chaque composante des $W^a_\mu$ et du spineur $\psi$ est $a + bt + cx + d\,tx$, où $a$, $b$, $c$, $d$ sont 32 paramètres libres. Le reste est encore nul (`reste nul = True`) : l'identité vaut pour toute valeur de ces paramètres.

Le terme d'auto-interaction extrait de $F^1$ vaut `-g*W2t*W3x + g*W3t*W2x`, c'est-à-dire $-g\,W^2_t W^3_x + g\,W^3_t W^2_x$ : ce sont les deux champs $W^2$ et $W^3$ qui composent la courbure du champ $W^1$. Pour les champs du carnet, il vaut $\dfrac{g}{8400}\left(-420 + 1365\,tx - 400\,t^3 + 336\,x^3 - 590\,t^2x^2\right)$.

Ce terme n'apparaît dans l'algèbre que parce que les générateurs ne commutent pas — c'est lui qui donne au Lagrangien de Yang–Mills $-\tfrac14 F^a_{\mu\nu}F^{a\,\mu\nu}$ son contenu propre, et qui interdit de traiter $SU(2)$ comme un simple « $U(1)$ en plus dimensionnel ».

*Le champ $f_{\mu\nu}$ se transforme comme un vecteur (« $f_{\mu\nu}$ transforms like a vector », copie archivée p. 193) : le terme bilinéaire est indispensable pour que $F^a_{\mu\nu}$ reste dans l'algèbre de Lie et se transforme dans la représentation adjointe. On le vérifie plus bas par le commutateur complet.*

## 7. Contrôle exact : $\mathrm{Tr}(F^2)$ invariant sous une jauge SU(2) explicite

**Hypothèse pré-enregistrée** : sous une transformation de jauge SU(2) engendrée par une phase $S = \exp(i\,\theta^a T^a)$ agissant sur des champs $W^a_\mu$ lisses, l'invariant $\mathrm{Tr}(F_{\mu\nu}F_{\mu\nu}^\dagger)$ est **inchangé** ; l'écart relatif maximal sur une grille de points doit rester $< 10^{-12}$.

Ici, les champs sont les trois $W^a_\mu$ de la section 6, et la rotation ci-dessous.

**La rotation, sans trigonométrie.** Une rotation SU(2) s'écrit $U = \exp(i\,\theta^a T^a)$ ; pour rester dans les fractions rationnelles, on la paramètre par un vecteur de Cayley : la direction $\vec n$ est donnée par projection stéréographique, $n = (2p,\, 2q,\, 1-p^2-q^2)/(1+p^2+q^2)$, et l'amplitude $k$ par $k = 2\tan(r/4)$, où $r$ est l'angle de rotation. La formule de Rodrigues prend alors la forme rationnelle

$$U = \frac{1-k^2/4}{1+k^2/4}\,I \;+\; \frac{2ik}{1+k^2/4}\; (n\cdot T),$$

qui est unitaire **par construction** : $(1-k^2/4)^2 + k^2/4 = (1+k^2/4)^2$ pour tout $k$. Les valeurs retenues sont $p = (2t+x)/5$, $q = (t-3x)/5 + 1/4$ et $k = (1+tx)/10$ : la direction et l'amplitude varient d'un point à l'autre, comme une vraie jauge locale ; le couplage vaut $g = 7/10$ et le point de contrôle $(37/100,\, 81/100)$.

**Point par point.** Les dérivées de $U$ sont celles d'un quotient de polynômes, donc exactes ; les produits matriciels restent dans les fractions rationnelles. Pour mesurer l'invariant en un point, chaque entrée est développée en série de Taylor **exacte à l'ordre 2** en $(\tau, \xi)$ autour de ce point, avec produits tronqués à l'ordre 2 : le terme constant du quotient est la valeur exacte au point, et les dérivées qui y entrent sont exactes. Le calcul est répété sur une grille $6\times 6$ de points.

In [8]:
// Controle SU(2) -- formes fermees (rotation de Rodrigues)
// On controle l'invariance de Tr(F F^dag) sous une jauge SU(2) explicite.
// Les champs sont ceux de la section 6 ; le couplage est fixe a g = 7/10.
var g_val = new Rat(7, 10);

// La rotation est decrite sans trigonometrie : direction
// n = (2p, 2q, 1 - p^2 - q^2) / (1 + p^2 + q^2) (projection stereographique de
// la sphere), et angle par k = 2 tan(r/4), ou r est l'angle de rotation.
Poly p = (2 * t + x) / 5;
Poly q_dir = (t - 3 * x) / 5 + new Rat(1, 4);
Poly k = (1 + t * x) / 10;
Poly norme = 1 + p * p + q_dir * q_dir;
var n = new Frac[] { new Frac(2 * p, norme), new Frac(2 * q_dir, norme), new Frac(1 - p * p - q_dir * q_dir, norme) };
var nT = Combinaison(n, demi);
Exige((n[0] * n[0] + n[1] * n[1] + n[2] * n[2] - 1).EstNulle,
    "n.n = 1 : la direction de la rotation est unitaire, identite exacte");

// U en forme fermee, identite de Rodrigues. Avec X = r n.T et T^a = sigma_a/2,
// on a (n.T)^2 = I/4, donc exp(iX) = cos(r/2) I + i sin(r/2) 2 n.T : le
// coefficient porte 2. Avec k = 2 tan(r/4) :
// cos(r/2) = (1 - k^2/4) / (1 + k^2/4), sin(r/2) = k / (1 + k^2/4).
Frac[][] Rodrigues(Poly kk, int coefficient)
{
    var k2 = kk * kk / 4;
    return M2.Echelle(new Frac(1, 1 + k2),
        M2.Add(M2.Echelle(1 - k2, M2.Identite), M2.Echelle(Cpx.I * coefficient * kk, nT)));
}

var U = Rodrigues(k, 2);
var Udag = M2.Dag(U);
Exige(M2.EstNulle(M2.Sub(M2.Mul(U, Udag), M2.Identite)),
    "U Udag = I : la rotation est unitaire, identite exacte en tout point");

// Point de controle et grille 6 x 6 : t et x de 0,1 a 1,0, en rationnels exacts.
var point_test = new Dictionary<string, Cpx> { ["t"] = new Rat(37, 100), ["x"] = new Rat(81, 100) };
var grille = new[] { new Rat(1, 10), new Rat(7, 25), new Rat(23, 50), new Rat(16, 25), new Rat(41, 50), Rat.One };

var ecart_unitarite = Pt.EcartMax(Pt.Mul(Pt.Evalue(U, point_test), Pt.Evalue(Udag, point_test)), Pt.Identite);
Console.WriteLine("Unitarite de U au point (0.37, 0.81) : ||U Udag - I|| = " + ecart_unitarite.Sci());

// Transformation de jauge : W'_mu = U W_mu U^dag + (i/g) (d_mu U) U^dag
Frac[][] Transforme(Frac[][] W_mu, Frac[][] matriceU, Frac[][] matriceUdag, int mu)
{
    var rotation = M2.Mul(M2.Mul(matriceU, W_mu), matriceUdag);
    var derivee = mu == 0 ? M2.DeriveeT(matriceU) : M2.DeriveeX(matriceU);
    return M2.Add(rotation, M2.Echelle(new Frac(Cpx.I, g_val), M2.Mul(derivee, matriceUdag)));
}

// W^a = Tr(sigma_a W) : les matrices de Pauli, de trace nulle et orthogonales
// deux a deux pour la trace, extraient les composantes.
Frac[] Composantes(Frac[][] matrice)
    => new[] { M2.Trace(M2.Mul(sigma_1, matrice)), M2.Trace(M2.Mul(sigma_2, matrice)), M2.Trace(M2.Mul(sigma_3, matrice)) };

// (1/2) somme_a (F^a)^2 en un point : Tr(F F^dag) = F^a F^b Tr(T^a T^b) = (1/2) somme_a (F^a)^2.
// Les F^a sont reels.
Rat Carre(Cpx z)
{
    if (!z.Im.EstNul) throw new InvalidOperationException("valeur non reelle");
    return z.Re * z.Re;
}
Rat Invariant(Frac[] Fc, IDictionary<string, Cpx> point)
    => (Carre(Fc[0].En(point)) + Carre(Fc[1].En(point)) + Carre(Fc[2].En(point))) / 2;

// --- Point par point ---
// Menee globalement, la chaine U W U^dag, puis ses derivees et ses produits,
// gonfle hors de mesure : chaque produit double le degre du denominateur et
// rien ne le reduit. On la menne donc point par point. Autour d'un point
// (t0, x0), chaque champ est remplace par son developpement de Taylor a l'ordre
// 2 en (tau, xi) = (t - t0, x - x0) : exact, puisque ce sont des polynomes et
// des fractions rationnelles de polynomes. Le pipeline de la section 6 lui est ensuite
// applique tel quel, tout produit etant tronque a l'ordre 2. Le terme constant
// est la valeur au point, le coefficient de tau (resp. xi) la derivee en t
// (resp. en x) : ce sont les derivees exactes, aucune difference finie.
var tau = Poly.Variable("tau", 0);
var xi = Poly.Variable("xi", 1);

Poly Puissance(Poly terme, int exposant)
{
    var resultat = Poly.Un;
    for (int e = 0; e < exposant; e++) resultat = resultat * terme;
    return resultat;
}

// Developpement de Taylor a l'ordre 2 d'un polynome en (t, x), ecrit en (tau, xi).
Poly Developpe(Poly champ, IDictionary<string, Cpx> point)
{
    var resultat = Poly.Zero;
    var derivee_t = champ;
    BigInteger factoriel_a = 1;
    for (int a = 0; a <= 2; a++)
    {
        var derivee = derivee_t;
        BigInteger factoriel_b = 1;
        for (int b = 0; a + b <= 2; b++)
        {
            var terme = derivee.En(point) / (Cpx)new Rat(factoriel_a * factoriel_b, BigInteger.One);
            resultat = resultat + (Poly)terme * Puissance(tau, a) * Puissance(xi, b);
            derivee = derivee.Dx;
            factoriel_b *= b + 1;
        }
        derivee_t = derivee_t.Dt;
        factoriel_a *= a + 1;
    }
    return resultat;
}

// Origine des coordonnees locales : (tau, xi) = (0, 0), c'est-a-dire le point lui-meme.
var origine = new Dictionary<string, Cpx> { ["tau"] = Cpx.Zero, ["xi"] = Cpx.Zero };

// Un champ, un vecteur de champs, une matrice de champs : remplaces par leurs developpements.
Frac Localise(Frac f, IDictionary<string, Cpx> point) => new Frac(Developpe(f.Num, point), Developpe(f.Den, point));
Frac[] Localise(Frac[] vecteur, IDictionary<string, Cpx> point)
    => vecteur.Select(f => Localise(f, point)).ToArray();
Frac[][] Localise(Frac[][] matrice, IDictionary<string, Cpx> point)
    => matrice.Select(ligne => ligne.Select(f => Localise(f, point)).ToArray()).ToArray();

var Wt_m = Combinaison(W_t, demi);
var Wx_m = Combinaison(W_x, demi);
var F_avant = ChampF(W_t, W_x, g_val);

var premier_point = new Dictionary<string, Cpx> { ["t"] = grille[0], ["x"] = grille[0] };
Console.WriteLine("Developpement de W^1_t = t/5 autour de (0.1, 0.1) : " + Localise(W_t[0], premier_point));

// Le pipeline de la section 6 sur des developpements locaux : tout produit y est tronque a l'ordre 2.
Frac[] EnOrdre2(Func<Frac[]> calcul)
{
    Poly.Ordre = 2;
    try { return calcul(); }
    finally { Poly.Ordre = -1; }
}

// Composantes de F au point, apres la transformation de jauge de matrice U.
Frac[] FApres(Frac[][] matriceU, IDictionary<string, Cpx> point)
{
    var u = Localise(matriceU, point);
    var ud = M2.Dag(u);
    return EnOrdre2(() => ChampF(Composantes(Transforme(Localise(Wt_m, point), u, ud, 0)),
                                 Composantes(Transforme(Localise(Wx_m, point), u, ud, 1)), g_val));
}

// Ecart relatif maximal sur la grille : |apres - avant| / |avant|, un rationnel
// exact. "apres" est calcule point par point et lu a l'origine de ses
// coordonnees locales ; "avant" est le champ polynomial de la section 6.
Rat EcartRelatifMax(Func<IDictionary<string, Cpx>, Frac[]> champApres)
{
    var max = Rat.Zero;
    foreach (var tt in grille)
        foreach (var xx in grille)
        {
            var point = new Dictionary<string, Cpx> { ["t"] = tt, ["x"] = xx };
            var avant = Invariant(F_avant, point);
            var apres = Invariant(champApres(point), origine);
            var ecart = (apres - avant).Abs / avant.Abs;
            if (ecart > max) max = ecart;
        }
    return max;
}

// Tout le reste du pipeline est identique d'une matrice a l'autre.
Rat EcartPour(Frac[][] matriceU) => EcartRelatifMax(point => FApres(matriceU, point));

var max_ecart = EcartPour(U);
Console.WriteLine("Invariance (1/2) sum (F^a)^2 = Tr(F F^dag) : ecart relatif max = " + max_ecart.Sci() + " sur 36 points");
Console.WriteLine("Verdict hypothese pre-enregistree (< 1e-12) : "
    + (max_ecart < new Rat(1, 1_000_000_000_000L) ? "CONFIRMEE" : "REJETEE"));
Exige(max_ecart.EstNul,
    "invariance de jauge SU(2) : Tr(F F^dag) invariant, ecart relatif exactement nul sur les 36 points");

OK   n.n = 1 : la direction de la rotation est unitaire, identite exacte


OK   U Udag = I : la rotation est unitaire, identite exacte en tout point


Unitarite de U au point (0.37, 0.81) : ||U Udag - I|| = 0.00e+00


Developpement de W^1_t = t/5 autour de (0.1, 0.1) : (1 + 10*tau) / 50


Invariance (1/2) sum (F^a)^2 = Tr(F F^dag) : ecart relatif max = 0.00e+00 sur 36 points


Verdict hypothese pre-enregistree (< 1e-12) : CONFIRMEE


OK   invariance de jauge SU(2) : Tr(F F^dag) invariant, ecart relatif exactement nul sur les 36 points


### Lecture du résultat

Trois mesures, trois lectures distinctes.

**L'unitarité** : `||U Udag - I|| = 0.00e+00` au point de contrôle. Ce zéro n'est pas un hasard numérique : la cellule a d'abord établi l'identité **exacte** $U U^\dagger = I$, valable en tout point et pour toute valeur de $k$ (`U Udag = I : la rotation est unitaire, identite exacte en tout point`). La paramétrisation de Cayley est unitaire par construction, pas par chance.

**L'invariance** : `ecart relatif max = 0.00e+00 sur 36 points`, et `Verdict hypothese pre-enregistree (< 1e-12) : CONFIRMEE`. Le zéro est là encore exact : l'écart est nul comme fraction rationnelle, il n'est pas simplement petit. L'invariant $\mathrm{Tr}(F F^\dagger) = \tfrac12\sum_a (F^a)^2$ ne bouge pas quand on change de jauge, et la grille $6\times 6$ couvre 36 points où la direction et l'amplitude de la rotation varient.

**Le sens du test.** Un zéro exact ne se suffit pas à lui-même : un calcul qui ne fait rien rendrait aussi zéro. Deux garde-fous l'encadrent. D'abord, les vérifications de la section 6 ont établi que $F$ est bien calculé à partir des champs $W$ — le calcul mord. Ensuite, la section 8 construit des transformations **fausses** et montre que l'écart, lui, n'est plus nul : c'est le contre-témoin qui donne son sens au zéro.

Enfin, la ligne de développement en un point rend le mécanisme lisible : `Developpement de W^1_t = t/5 autour de (0.1, 0.1) : (1 + 10*tau) / 50`, c'est-à-dire $\tfrac1{50} + \tfrac{\tau}{5}$ — la valeur au point, plus la dérivée exacte multipliée par l'écart. C'est cette exactitude locale, répétée sur les 36 points, qui rend la mesure concluante.

## 8. Contre-témoin : une transformation fausse doit être détectée

Un test qui ne sait pas dire non ne prouve rien. Construisons deux transformations **volontairement fausses** et vérifions que le calcul les **détecte**.

**Premier contre-témoin.** La transformation de jauge abélienne seule : $W^a_\mu \to W^a_\mu + \partial_\mu\theta^a$ (le terme $\varepsilon^{abc}$ est omis). Seule la règle de transformation change : mêmes paramètres $\theta^a = k\,n^a$, même grille, même calcul de $F$.

**Seconde contre-épreuve.** La rotation est prise avec un coefficient faux dans la formule de Rodrigues : $U = (1-k^2/4)/(1+k^2/4)\,I + \frac{ik}{1+k^2/4}(n\cdot T)$ au lieu de $\frac{2ik}{1+k^2/4}$ — le facteur 2 est omis. Cette matrice n'est plus unitaire, et l'invariance doit se briser. Cette seconde contre-épreuve est propre au jumeau : le carnet Python ne teste que la transformation abélienne.

In [9]:
// Contre-temoin : la transformation abelienne appliquee au champ non abelien.
// On garde la meme rotation (donc les memes theta^a) et la meme grille a la section 7 ;
// seule la regle de transformation change : simple translation
// W_mu -> W_mu + (1/g) d_mu theta, sans la rotation adjointe U W_mu U^dag.
// Tout le reste est identique : memes developpements locaux, meme pipeline.

// theta^a = k n^a : direction n, amplitude k (k = 2 tan(r/4)).
var theta = new Frac[] { n[0] * k, n[1] * k, n[2] * k };

Frac[] FAbelien(IDictionary<string, Cpx> point)
{
    var theta_local = Localise(theta, point);
    var Wt_ct = new Frac[3];
    var Wx_ct = new Frac[3];
    for (int i = 0; i < 3; i++)
    {
        Wt_ct[i] = Localise(W_t[i], point) + theta_local[i].Dt / g_val;
        Wx_ct[i] = Localise(W_x[i], point) + theta_local[i].Dx / g_val;
    }
    return EnOrdre2(() => ChampF(Wt_ct, Wx_ct, g_val));
}

var max_ct = EcartRelatifMax(FAbelien);
Console.WriteLine("Contre-temoin abelien : ecart relatif max = " + max_ct.Sci() + " (attendu >> 1e-6)");
Console.WriteLine("La transformation incomplete est DETECTEE : " + (max_ct > new Rat(1, 1_000_000L) ? "OUI" : "NON"));
Exige(max_ct > new Rat(1, 1_000_000L),
    "contre-temoin : la regle abelienne seule rompt l'invariance -- l'ecart est detecte (seuil 1e-6)");

// Second contre-temoin : la forme de Rodrigues avec le coefficient 1 au lieu de
// 2 devant k n.T. C'est le piege sin(r/2)/r de la source : il se paie par une
// matrice non unitaire, puis par une invariance rompue. Seule la matrice U change.
var U_piege = Rodrigues(k, 1);
var ecart_piege_U = Pt.EcartMax(
    Pt.Mul(Pt.Evalue(U_piege, point_test), Pt.Evalue(M2.Dag(U_piege), point_test)), Pt.Identite);
var max_piege = EcartPour(U_piege);
Console.WriteLine("Contre-epreuve, coefficient 1 au lieu de 2 dans U : ||U Udag - I|| = " + ecart_piege_U.Sci()
    + " ; ecart relatif max = " + max_piege.Sci());
Exige(!ecart_piege_U.EstNul && max_piege > new Rat(1, 1_000_000L),
    "contre-epreuve : un coefficient de Rodrigues faux est detecte (U non unitaire, invariance rompue)");

Contre-temoin abelien : ecart relatif max = 1.21e-01 (attendu >> 1e-6)


La transformation incomplete est DETECTEE : OUI


OK   contre-temoin : la regle abelienne seule rompt l'invariance -- l'ecart est detecte (seuil 1e-6)


Contre-epreuve, coefficient 1 au lieu de 2 dans U : ||U Udag - I|| = 1.26e-02 ; ecart relatif max = 1.06e-01


OK   contre-epreuve : un coefficient de Rodrigues faux est detecte (U non unitaire, invariance rompue)


### Lecture du résultat

**Premier contre-témoin** : `ecart relatif max = 1.21e-01`, soit environ $12\%$ — très au-dessus du seuil $10^{-6}$ (`La transformation incomplete est DETECTEE : OUI`). Oublier le terme $\varepsilon^{abc}$ casse effectivement l'invariance, et le calcul le voit. Le contraste avec le zéro exact de la section 7 est net : le test distingue une vraie transformation d'une fausse.

**Seconde contre-épreuve** : `||U Udag - I|| = 1.26e-02` et `ecart relatif max = 1.06e-01`. La matrice au coefficient faux n'est pas unitaire — l'écart d'unitarité est non nul — et l'invariance se rompt à environ $11\%$. Les deux niveaux de contrôle sont cohérents : une rotation fausse est à la fois détectée comme non unitaire et comme non invariante.

**Ce que cela établit.** Les deux zéros de la section 7 ne sont pas l'effet d'un calcul inerte : les mêmes fonctions, appliquées à des transformations incorrectes, produisent des écarts de l'ordre de $10^{-1}$. Le test a donc un pouvoir de détection mesuré, sur les deux plans où une erreur pouvait se glisser — la règle de transformation des champs et la matrice de rotation elle-même.

## 9. Exercices

### Exercice 1 — Lagrangien de Dirac invariant de jauge U(1)

Reprendre la construction des sections 1 à 4 pour un **champ de Dirac** en $1+1$ dimensions, avec les matrices $\gamma^0 = \begin{pmatrix}0 & I\\ I & 0\end{pmatrix}$ et $\gamma^1 = \begin{pmatrix}0 & -I\\ I & 0\end{pmatrix}$ (représentation chirale). Écrire le spineur comme une colonne de `Frac`, et vérifier que $\bar\psi = \psi^\dagger\gamma^0$ et le lagrangien $\bar\psi\,(i\gamma^\mu D_\mu - m)\,\psi$ sont invariants sous la jauge locale $U(1)$.

In [10]:
// Exercice 1 -- a completer
// Indice : gamma^0 = [[0, i],[i, 0]], gamma^1 = [[0, -i],[i, 0]] en representation chirale 1+1 D
// (ici i = Cpx.I, et une matrice est un tableau imbrique de Frac).
// Etape 1 : construire psi comme vecteur a 2 composantes complexes (Frac[]).
// Etape 2 : former D_mu psi avec le A_mu du carnet.
// Etape 3 : verifier que la transformation conjointe laisse L invariant (reste nul, Exige).
Func<bool> resultat_ex1 = null;   // TODO etudiant

Console.WriteLine("Exercice a completer : jauge U(1) du champ de Dirac");

Exercice a completer : jauge U(1) du champ de Dirac


### Exercice 2 — Doublet scalaire SU(2) et dérivée covariante matricielle

Reprendre les sections 5 à 7 pour un **doublet scalaire complexe** $\Phi = (\phi_1, \phi_2)^T$ dont chaque composante est un champ complexe du type de ceux de la section 1. La dérivée covariante devient $D_\mu\Phi = \partial_\mu\Phi + ig\,W^a_\mu T^a\Phi$, avec $T^a = \sigma^a/2$. Vérifier que le lagrangien $(D_\mu\Phi)^\dagger(D^\mu\Phi) - m^2\Phi^\dagger\Phi$ est invariant sous la transformation locale $\Phi' = S\Phi$, $W'_\mu = S W_\mu S^\dagger + \frac{i}{g}(\partial_\mu S)S^\dagger$, où $S$ est une rotation de Rodrigues de la section 7 (direction et amplitude variables).

In [11]:
// Exercice 2 -- a completer
// Indice : Phi' = S Phi, W'_mu = S W_mu S^dag + (i/g) (d_mu S) S^dag
// Etape 1 : poser S = exp(i theta^a T^a) avec les theta du carnet (rotation de Rodrigues de la section 7).
// Etape 2 : developper (D Phi)^dag (D Phi) avant/apres, puis comparer les deux.
Func<bool> resultat_ex2 = null;   // TODO etudiant

Console.WriteLine("Exercice a completer : doublet scalaire SU(2)");

Exercice a completer : doublet scalaire SU(2)


### Exercice 3 — Composante $F^3$ et limite abélienne

Calculer la composante $F^3_{tx}$ de la courbure pour les champs $W$ de la section 6 :

$$F^3_{tx} = \partial_t W^3_x - \partial_x W^3_t - g\left(W^1_t W^2_x - W^2_t W^1_x\right).$$

Vérifier que le résultat coïncide avec celui du calcul matriciel de la section 6, puis montrer que pour $W^1_t = W^2_x = 0$, le terme d'auto-interaction se réduit à $+g\,W^2_t W^1_x$ et que la limite $g \to 0$ redonne le tenseur abélien de la section 4.

In [12]:
// Exercice 3 -- a completer
// Indice : F^3 = d_t W^3_x - d_x W^3_t - g (W^1_t W^2_x - W^2_t W^1_x)
// Etape 1 : poser les fonctions (champs W de la section 6), calculer la formule.
// Etape 2 : comparer au commutateur [D_t, D_x] de la section 6.
Func<bool> resultat_ex3 = null;   // TODO etudiant

Console.WriteLine("Exercice a completer : composante F^3 et limite abelienne");

Exercice a completer : composante F^3 et limite abelienne


## 10. Périmètre honnête

**Ce qui est établi.** Les identités des sections 1 à 6 sont exactes, sur les champs polynomiaux du carnet : le calcul manipule des polynômes à coefficients rationnels et conclut au polynôme nul. La section 6 est en outre établie sur des champs à **coefficients symboliques** : 32 paramètres libres, identité valable pour toute valeur de ces paramètres. La section 7 est un contrôle exact **point par point** sur une grille de 36 points, avec développements de Taylor exacts à l'ordre 2 : ce n'est ni une preuve générale, ni un balayage exhaustif — c'est un contrôle de cohérence interne à un point de contrôle, complété par les contre-témoins de la section 8.

**Ce qui n'est pas établi.** Rien ici ne touche au problème du **gap de masse** (prix du millénaire, Clay Institute ; voir aussi Jaffe–Witten, « Quantum Yang–Mills Theory », copie archivée au gisement) : cet arc traite l'**invariance de jauge classique**, pas la théorie quantique en 4 dimensions. Aucune affirmation sur l'existence d'une théorie quantique de Yang–Mills n'est faite ni suggérée.

**Champs.** Les sections 1 à 4 valent pour les champs polynomiaux du carnet ; la masse $m$, la charge $q$ et la phase restent des paramètres libres. La section 6 vaut sur champs symboliques, mais toujours dans la classe « chaque composante est un polynôme de degré total au plus 2 » — une identité polynomiale qui vaut sur cette classe vaut sur tous les champs polynomiaux par linéarité, mais pas sur des champs arbitraires non polynomiaux.

**Rotations.** La paramétrisation de Cayley $k = 2\tan(r/4)$ couvre les rotations d'angle $r \in [0, 2\pi[$ ; elle n'atteint pas $U = -I$ (qui correspond à $k$ infini, angle $2\pi$). Le contrôle de la section 7 porte donc sur cette famille, pas sur tout $SU(2)$.

**Écarts avec le carnet Python.** Le carnet Python travaille en flottants et en calcul formel symbolique ; les écarts numériques qu'il rapporte (par exemple $3.12 \times 10^{-18}$ et $3.92 \times 10^{-15}$ pour les contrôles d'invariance, $8.16 \times 10^{-1}$ pour le contre-témoin) deviennent ici des zéros exacts et un écart de $1.21 \times 10^{-1}$ — même verdict physique, arithmétique différente. La contre-épreuve de Rodrigues est un ajout du jumeau — le carnet Python ne teste que la transformation abélienne ; ici elle est **exécutée** : $1.26 \times 10^{-2}$ d'écart d'unitarité et $1.06 \times 10^{-1}$ d'écart d'invariance.

**Suite.** Le carnet [Jauge-03](Jauge-03-Action-Wilson-Confinement-Python.ipynb) passe de l'invariance locale au **confinement** : l'action de Wilson sur réseau, et la boucle de Wilson comme observable de confinement.

## À retenir

| Notion | Section | Résultat du jumeau C# |
|---|---|---|
| Symétrie globale U(1) | 1 | $\delta L = 0$ exact (polynôme nul) |
| Phase locale : l'invariance se brise | 2 | $\delta L \neq 0$, identique à la forme fermée ; nul dès que $\partial_\mu\alpha = 0$ |
| Dérivée covariante | 3 | $D'_\mu\varphi' = e^{i\alpha}D_\mu\varphi$ exact, en $t$ et en $x$ |
| Tenseur $F_{\mu\nu}$ abélien | 4 | $F' - F = 0$ ; $\delta L_{\text{total}} = 0$ exact |
| Algèbre SU(2) | 5 | $[T^a,T^b] = i\varepsilon^{abc}T^c$ : 9/9 paires vérifiées |
| Courbure non abélienne | 6 | $[D_t,D_x] = igF^aT^a$ exact, sur 32 coefficients symboliques libres |
| Invariance $\mathrm{Tr}(F^2)$ | 7 | écart nul sur 36 points, hypothèse pré-enregistrée CONFIRMÉE |
| Contre-témoin abélien | 8 | écart $1{,}21 \times 10^{-1}$ — transformation fausse DÉTECTÉE |
| Contre-épreuve Rodrigues | 8 | $||UU^\dagger - I|| = 1{,}26 \times 10^{-2}$, écart $1{,}06 \times 10^{-1}$ |

Le fil : une symétrie **globale** impose un courant conservé ; exiger la symétrie **en chaque point** force un champ de jauge, dont la propre dynamique est invariante — et, en SU(2), ce champ **s'auto-interagit**, parce que les générateurs ne commutent pas. Tout se vérifie exactement, et les contre-témoins montrent que le calcul sait dire non.

## Vérifiez votre compréhension

1. **Symétrie globale vs locale.** Pourquoi le lagrangien d'un champ scalaire complexe est-il invariant sous une phase constante, et pourquoi cesse-t-il de l'être dès que la phase dépend de $(t,x)$ ? Quelle est la seule quantité dont dépend alors l'écart ?
   <details><summary>Réponse</summary>La phase constante disparaît dans $e^{i\alpha}e^{-i\alpha}$ (terme de masse) et sa dérivée est nulle (terme cinétique). Pour une phase locale, l'écart ne dépend que de $\partial_\mu\alpha$ : c'est exactement ce que montre la section 2, où l'écart redevient nul quand les dérivées de $\alpha$ sont neutralisées.</details>

2. **Dérivée covariante.** Quelle transformation de $A_\mu$ compense l'apparition de $\partial_\mu\alpha$ dans $D_\mu\varphi$, et pourquoi le signe de cette transformation est-il imposé ?
   <details><summary>Réponse</summary>$A_\mu \to A_\mu - \frac1q\partial_\mu\alpha$. Le terme $iqA_\mu\varphi$ gagne alors $-i\partial_\mu\alpha\,\varphi$, qui compense exactement le $+i\partial_\mu\alpha\,\varphi$ apporté par la dérivée de la phase ; les deux effets se compensent dans $\delta L$, d'où l'invariance exacte de la section 3.</details>

3. **Ce que mesure un contre-témoin.** Dans la section 8, quelle est la valeur de l'écart relatif maximal pour la transformation abélienne incomplète, et que prouve-t-elle ?
   <details><summary>Réponse</summary>$1{,}21 \times 10^{-1}$, très au-dessus du seuil $10^{-6}$. Elle prouve que le test détecte une transformation fausse : le zéro exact de la section 7 n'est pas l'effet d'un calcul inerte.</details>

4. **Seuil et pouvoir de détection.** Pourquoi l'hypothèse pré-enregistrée exige-t-elle un écart $< 10^{-12}$ plutôt qu'un simple « petit » ?
   <details><summary>Réponse</summary>Le calcul est exact : un écart non nul serait un écart de structure, pas d'arrondi. Le seuil $10^{-12}$ sert de garde contre une erreur d'implémentation (dérivées fausses, troncature incorrecte) tout en restant très en dessous des écarts produits par les transformations fausses ($10^{-1}$), ce qui rend la détection non ambiguë.</details>

## Pour aller plus loin

- **[Jauge-03 — Action de Wilson et confinement](Jauge-03-Action-Wilson-Confinement-Python.ipynb)** (Python) : l'étape suivante de la série — l'invariance locale posée ici devient l'action de Wilson sur réseau, et la boucle de Wilson l'observable du confinement.
- **[Jauge-01 — Théorème de Noether](Jauge-01-Theoreme-Noether-Python.ipynb)** (Python) : l'étape précédente — de la symétrie globale au courant conservé.
- **Yang & Mills (1954)**, *Conservation of Isotopic Spin and Isotopic Gauge Invariance* : l'article fondateur, dont la copie archivée est citée en Références — à lire pour la construction originale du tenseur $f_{\mu\nu}$ et la discussion du terme bilinéaire.
- **Jaffe & Witten**, *Quantum Yang–Mills Theory* : l'énoncé du problème du gap de masse, pour situer ce que ce carnet ne traite pas.

**Fil de la série** : [#23161 — EPIC Série Physique numérique](https://github.com/jsboige/CoursIA/issues/23161)

## Références

- C. N. Yang et R. L. Mills, « Conservation of Isotopic Spin and Isotopic Gauge Invariance », *Physical Review* **96**, 191–195 (1954) — copie archivée au gisement de bibliographie partagé.
- A. Jaffe et E. Witten, « Quantum Yang–Mills Theory », description du problème du prix du millénaire, Clay Mathematics Institute (2000) — copie archivée au gisement de bibliographie partagé.
- Carnet source Python : [Jauge-02 — De la symétrie globale à la jauge locale : U(1) puis SU(2)](Jauge-02-Invariance-Jauge-U1-SU2-Python.ipynb).
- [Jauge-01 — Théorème de Noether](Jauge-01-Theoreme-Noether-Python.ipynb) · [Jauge-03 — Action de Wilson et confinement](Jauge-03-Action-Wilson-Confinement-Python.ipynb) · [Index de la série](README.md).